In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.nn.init as init
import scipy.linalg as sci
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt

In [63]:
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

True


In [64]:
## Learning parameters
initial_run = 1 #1: starts training from scratch; 0: resumes training 
n_epochs = 5000 #Number of training epochs, for observing the current performance set it to 0
learning_rate = 1e-4 #Learning rate

batch_size = 1024 #Mini-batch size
test_size = 5000 #Size of the validation/test set
batch_per_epoch = 10 #Numbers of mini-batches per epoch

anneal_param = 1.0 #Initial annealing parmeter
annealing_rate = 1.001 #Annealing rate

In [65]:
## System parameters
M = 64 #Number of BS antennas
P = 1 #Power
K =  2 #Number of users
L = 8 #Number of pilots
Lp = [2, 3, 4, 5, 6, 7, 8] #Number of paths
B = [1, 3, 6, 15, 20, 30, 40, 50, 60] #Number of feedback bits per user

## Limited scattering channel parameters
LSF_UE = np.array([0.0,0.0],dtype=np.float32) #Mean of path gains for K users
Mainlobe_UE= np.array([0,0],dtype=np.float32) #Center of the AoD range for K users
HalfBW_UE = np.array([30.0,30.0],dtype=np.float32) #Half of the AoD range for K users

# SNR
snr_dl = 10 #SNR in dB
noise_std_dl = np.float32(np.sqrt(1/2)*np.sqrt(P/10**(snr_dl/10))) #STD of the Gaussian noise (per real dim.)

### Batch data 생성 함수

In [66]:
## 리스트 인덱싱은 [층, 행, 열], [행, 열]임!
def generate_batch_data(batch_size,M,K,
                        Lp,#number of paths
                        LSF_UE #Mean of path gains for K users
                        ,Mainlobe_UE #Center of the AoD range for K users
                        ,HalfBW_UE #Half of the AoD range for K users
                        ):
    alphaR_input = np.zeros((batch_size,Lp,K))
    alphaI_input = np.zeros((batch_size,Lp,K))
    theta_input = np.zeros((batch_size,Lp,K))
    for kk in range(K): # for the number of users
        alphaR_input[:,:,kk] = np.random.normal(loc=LSF_UE[kk], scale=1.0/np.sqrt(2), size=[batch_size,Lp])
        alphaI_input[:,:,kk] = np.random.normal(loc=LSF_UE[kk], scale=1.0/np.sqrt(2), size=[batch_size,Lp])
        theta_input[:,:,kk] = np.random.uniform(low=Mainlobe_UE[kk]-HalfBW_UE[kk], high=Mainlobe_UE[kk]+HalfBW_UE[kk], size=[batch_size,Lp])
 
    #### Actual Channel
    from0toM = np.float32(np.arange(0, M, 1))
    alpha_act = alphaR_input + 1j*alphaI_input
    theta_act = (np.pi/180)*theta_input
    
    h_act = np.complex64(np.zeros((batch_size,M,K)))
    hR_act = np.float32(np.zeros((batch_size,M,K)))
    hI_act = np.float32(np.zeros((batch_size,M,K)))
    
    for kk in range(K):
        for ll in range(Lp):
            theta_act_expanded_temp = np.tile(np.reshape(theta_act[:,ll,kk],[-1,1]),(1,M))
            response_temp = np.exp(1j*np.pi*np.multiply(np.sin(theta_act_expanded_temp),from0toM))
            alpha_temp = np.reshape(alpha_act[:,ll,kk],[-1,1])
            h_act[:,:,kk] += (1/np.sqrt(Lp))*alpha_temp*response_temp
        hR_act[:,:,kk] = np.real(h_act[:,:,kk])
        hI_act[:,:,kk] = np.imag(h_act[:,:,kk])
    
    h_act = torch.tensor(h_act, dtype = torch.complex64).to(device)
    hR_act = torch.tensor(hR_act, dtype = torch.float32).to(device)
    hI_act = torch.tensor(hI_act, dtype = torch.float32).to(device)
        
    return(h_act, hR_act, hI_act)

In [67]:
# DNN을 통과한 pilot output -> OMP
def OMPce_Rnd_pilot(h_act, # (batch, M, K)
                                    pilot_length, # L
                                    num_users, num_tantenna,  Mainlobe_UE, HalfBW_UE, noise_std, sparsity_level, dict_size):
    AoD_low = Mainlobe_UE[0]-HalfBW_UE[0]
    AoD_high = Mainlobe_UE[0]+HalfBW_UE[0]
    
    # generate random pilot sequence for each user at each batch data, identical for each user
    pilot_sequence = torch.sqrt(torch.tensor(2)) * torch.randn([num_tantenna, pilot_length], dtype = torch.complex64).to(device) # (M, L)
    pilot_sequence = pilot_sequence / torch.sqrt(torch.real(torch.einsum('ii', (pilot_sequence.H @ pilot_sequence)))).to(device)
    pilot_output = torch.matmul(h_act.mH , pilot_sequence).to(device) + torch.randn([h_act.shape[0], h_act.shape[2], pilot_sequence.shape[1]], dtype = torch.complex64).to(device) * torch.tensor(noise_std).to(device)
    # pilot output of shape (batch, K, L)
    
    #### Generating S
    index_M = torch.arange(num_tantenna, dtype = torch.float32).to(device)
    theta_vec = torch.linspace(AoD_low, AoD_high, dict_size).to(device)
    theta_vec_rad = (torch.pi / 180) * theta_vec
    theta_vec_sin = torch.sin(theta_vec_rad).to(device)
    S = torch.ones([dict_size, num_tantenna], dtype = torch.complex64).to(device)  # (dictsize, M)
    for ss in range(dict_size):
        S[ss, :] = S[ss, :] * torch.exp(1j * torch.pi * index_M * theta_vec_sin[ss])
    Phi = pilot_sequence.T @ S.T # (L, dict_size)
    
    ### Parallel computation ############################################################################
    Y = pilot_output # (batch, K, L)
    for kk in range(num_users):
        u = Y[:, kk, :].unsqueeze(1).mT # (batch, L, 1)
        v = torch.clone(u) # residual
        
        for tt in range(sparsity_level):
            absPhi_H_v = abs(torch.matmul(Phi.H.unsqueeze(0), v)).squeeze(-1) # (batch, dict_size)
            idx = torch.argmax(absPhi_H_v, dim = 1).unsqueeze(-1) if tt == 0 else torch.cat((idx, torch.argmax(absPhi_H_v, dim = 1).unsqueeze(-1)), dim = 1)
            phi_s = Phi[:, idx].permute(1,0,2) if  tt == 0 else Phi[:, idx].permute(1,0,2) # (batch, L, indices)
            b = torch.bmm(torch.linalg.pinv(phi_s), u)
            v = v - torch.bmm(phi_s, b) # update residual
        
        hhat = torch.bmm(b.mT, S[idx, :]) # (batch, 1, M)
        H_hat = hhat if kk == 0 else torch.cat((H_hat, hhat), dim = 1) # (batch, K, M) estimating H^H
        alphaR = torch.real(b) if kk == 0 else torch.cat((alphaR, torch.real(b)), dim=2)
        alphaI = torch.imag(b) if kk == 0 else torch.cat((alphaI, torch.imag(b)), dim=2)
        theta = theta_vec[idx].unsqueeze(-1) if kk == 0 else torch.cat((theta, theta_vec[idx].unsqueeze(-1)), dim=2)
    return H_hat.to(device), alphaR.to(device), alphaI.to(device), theta.to(device)      

In [68]:
def rate_calc(K, V, h_act, noise_std):
    hH_V = torch.bmm(h_act.mH, V).to(device)
    hH_V = torch.pow(abs(hH_V), 2).to(device)
    noise_std = torch.tensor(noise_std).to(device)
    
    rate = torch.zeros(h_act.shape[0]).to(device)
    for kk in range(K):
        sig_pow = torch.sum(hH_V[:, kk, :], dim=-1)
        wanted_sig = hH_V[:, kk, kk]
        rate += torch.log2(1 + ( wanted_sig / (sig_pow - wanted_sig + 2 * noise_std ** 2)) )

    sumrate = torch.mean( rate ).to(device)

    return sumrate

In [69]:
import pickle
with open("Lloyd-Max quantizer boundaries", 'rb') as fr:
    L_quantizer_boundary = pickle.load(fr)
with open("Lloyd-Max quantizer levels", 'rb') as fr:
    L_quantizer_level = pickle.load(fr)
with open("Uniform quantizer boundaries", 'rb') as fr:
    U_quantizer_boundary = pickle.load(fr)
with open("Uniform quantizer levels", 'rb') as fr:
    U_quantizer_level = pickle.load(fr)

In [70]:
def quantizer(R, I, T, Q_levels, L_bound, L_level, U_bound, U_level):
    Q_R = torch.zeros_like(R, dtype=torch.float32).to(device)
    Q_I = torch.zeros_like(I, dtype=torch.float32).to(device)
    Q_T = torch.zeros_like(T, dtype=torch.float32).to(device)
    
    for qq in range(Q_levels):
        ### condition for alpha_R
        if qq == 0:
            mask = (R <= L_bound[0])
        elif qq == Q_levels-1:
            mask = (R >= L_bound[qq-1])
        else:
            mask = (R <= L_bound[qq]) & (R >= L_bound[qq-1])
        Q_R[mask] = L_level[qq]
        
        ### condition for alpha_I
        if qq == 0:
            mask = (I <= L_bound[0])
        elif qq == Q_levels-1:
            mask = (I >= L_bound[qq-1])
        else:
            mask = (I <= L_bound[qq]) & (I >= L_bound[qq-1])
        Q_I[mask] = L_level[qq]
        
        ### condition for theta
        mask = (T <= U_bound[qq+1]) & (T >= U_bound[qq])
        Q_T[mask] = U_level[qq]
        
    return Q_R, Q_I, Q_T

In [83]:
iter = 10000
sumrate_ZF = []
sumrate_MRT = []
ZF_lim = []
MRT_lim = []
for ll in range(len(Lp)):
    paths = Lp[ll] 
    h_act, hr_act, hi_act = generate_batch_data(iter, M, K, paths, LSF_UE, Mainlobe_UE, HalfBW_UE)
        ## OMP channel estimation with infinite rate feedback
    H_hat, alphaR, alphaI, theta = OMPce_Rnd_pilot(h_act, L, K, M,  Mainlobe_UE, HalfBW_UE, noise_std_dl, paths, 1000) ## (batch, K, M) estimated H^H
    if ll == 0 or ll == 3: # Lp=2 -> 5bit씩, Lp=5 -> 2bit씩
        Q_bit = int( 10 / paths )
        Q_level = 2 ** Q_bit
        Q_alphaR, Q_alphaI, Q_theta = quantizer(alphaR, alphaI, theta, Q_level, 
                                                L_quantizer_boundary[Q_level], L_quantizer_level[Q_level], U_quantizer_boundary[2**Q_bit], U_quantizer_level[2**Q_bit])
    elif ll == 1: # Lp=3 -> 3+3+4 bit
        alphaR_1 = alphaR[:, :2, :]
        alphaR_2 = alphaR[:, 2:, :]
        alphaI_1 = alphaI[:, :2, :]
        alphaI_2 = alphaI[:, 2:, :]
        theta_1 = theta[:, :2, :]
        theta_2 = theta[:, 2:, :]
        Q_bit_1 = 3
        Q_bit_2 = 4
  
        Q_alphaR_1, Q_alphaI_1, Q_theta_1 = quantizer(alphaR_1, alphaI_1, theta_1, 2**Q_bit_1, 
                                                L_quantizer_boundary[2**Q_bit_1], L_quantizer_level[2**Q_bit_1], U_quantizer_boundary[2**Q_bit_1], U_quantizer_level[2**Q_bit_1])
        Q_alphaR_2, Q_alphaI_2, Q_theta_2 = quantizer(alphaR_2, alphaI_2, theta_2, 2**Q_bit_2, 
                                                L_quantizer_boundary[2**Q_bit_2], L_quantizer_level[2**Q_bit_2], U_quantizer_boundary[2**Q_bit_2], U_quantizer_level[2**Q_bit_2])
        Q_alphaR = torch.cat((Q_alphaR_1, Q_alphaR_2), dim=1)
        Q_alphaI = torch.cat((Q_alphaI_1, Q_alphaI_2), dim=1)
        Q_theta = torch.cat((Q_theta_1, Q_theta_2), dim=1)
        
    elif ll == 2: # Lp=4 -> 3*2 + 2*2 bit
        alphaR_1 = alphaR[:, :2, :]
        alphaR_2 = alphaR[:, 2:, :]
        alphaI_1 = alphaI[:, :2, :]
        alphaI_2 = alphaI[:, 2:, :]
        theta_1 = theta[:, :2, :]
        theta_2 = theta[:, 2:, :]
        Q_bit_1 = 3
        Q_bit_2 = 2
  
        Q_alphaR_1, Q_alphaI_1, Q_theta_1 = quantizer(alphaR_1, alphaI_1, theta_1, 2**Q_bit_1, 
                                                L_quantizer_boundary[2**Q_bit_1], L_quantizer_level[2**Q_bit_1], U_quantizer_boundary[2**Q_bit_1], U_quantizer_level[2**Q_bit_1])
        Q_alphaR_2, Q_alphaI_2, Q_theta_2 = quantizer(alphaR_2, alphaI_2, theta_2, 2**Q_bit_2, 
                                                L_quantizer_boundary[2**Q_bit_2], L_quantizer_level[2**Q_bit_2], U_quantizer_boundary[2**Q_bit_2], U_quantizer_level[2**Q_bit_2])
        Q_alphaR = torch.cat((Q_alphaR_1, Q_alphaR_2), dim=1)
        Q_alphaI = torch.cat((Q_alphaI_1, Q_alphaI_2), dim=1)
        Q_theta = torch.cat((Q_theta_1, Q_theta_2), dim=1)
        print(Q_alphaR.shape)
        
    elif ll == 4: # Lp=6 -> 1*2 + 2*4 bit
        alphaR_1 = alphaR[:, :2, :]
        alphaR_2 = alphaR[:, 2:, :]
        alphaI_1 = alphaI[:, :2, :]
        alphaI_2 = alphaI[:, 2:, :]
        theta_1 = theta[:, :2, :]
        theta_2 = theta[:, 2:, :]
        Q_bit_1 = 1
        Q_bit_2 = 2
  
        Q_alphaR_1, Q_alphaI_1, Q_theta_1 = quantizer(alphaR_1, alphaI_1, theta_1, 2**Q_bit_1, 
                                                L_quantizer_boundary[2**Q_bit_1], L_quantizer_level[2**Q_bit_1], U_quantizer_boundary[2**Q_bit_1], U_quantizer_level[2**Q_bit_1])
        Q_alphaR_2, Q_alphaI_2, Q_theta_2 = quantizer(alphaR_2, alphaI_2, theta_2, 2**Q_bit_2, 
                                                L_quantizer_boundary[2**Q_bit_2], L_quantizer_level[2**Q_bit_2], U_quantizer_boundary[2**Q_bit_2], U_quantizer_level[2**Q_bit_2])
        Q_alphaR = torch.cat((Q_alphaR_1, Q_alphaR_2), dim=1)
        Q_alphaI = torch.cat((Q_alphaI_1, Q_alphaI_2), dim=1)
        Q_theta = torch.cat((Q_theta_1, Q_theta_2), dim=1)
        
    elif ll == 5: # Lp=7 -> 1*4 + 2*3 bit
        alphaR_1 = alphaR[:, :4, :]
        alphaR_2 = alphaR[:, 4:, :]
        alphaI_1 = alphaI[:, :4, :]
        alphaI_2 = alphaI[:, 4:, :]
        theta_1 = theta[:, :4, :]
        theta_2 = theta[:, 4:, :]
        Q_bit_1 = 1
        Q_bit_2 = 2
  
        Q_alphaR_1, Q_alphaI_1, Q_theta_1 = quantizer(alphaR_1, alphaI_1, theta_1, 2**Q_bit_1, 
                                                L_quantizer_boundary[2**Q_bit_1], L_quantizer_level[2**Q_bit_1], U_quantizer_boundary[2**Q_bit_1], U_quantizer_level[2**Q_bit_1])
        Q_alphaR_2, Q_alphaI_2, Q_theta_2 = quantizer(alphaR_2, alphaI_2, theta_2, 2**Q_bit_2, 
                                                L_quantizer_boundary[2**Q_bit_2], L_quantizer_level[2**Q_bit_2], U_quantizer_boundary[2**Q_bit_2], U_quantizer_level[2**Q_bit_2])
        Q_alphaR = torch.cat((Q_alphaR_1, Q_alphaR_2), dim=1)
        Q_alphaI = torch.cat((Q_alphaI_1, Q_alphaI_2), dim=1)
        Q_theta = torch.cat((Q_theta_1, Q_theta_2), dim=1)
        
    else: # Lp=8 -> 1*6 + 2*2 bit
        alphaR_1 = alphaR[:, :6, :]
        alphaR_2 = alphaR[:, 6:, :]
        alphaI_1 = alphaI[:, :6, :]
        alphaI_2 = alphaI[:, 6:, :]
        theta_1 = theta[:, :6, :]
        theta_2 = theta[:, 6:, :]
        Q_bit_1 = 1
        Q_bit_2 = 2
  
        Q_alphaR_1, Q_alphaI_1, Q_theta_1 = quantizer(alphaR_1, alphaI_1, theta_1, 2**Q_bit_1, 
                                                L_quantizer_boundary[2**Q_bit_1], L_quantizer_level[2**Q_bit_1], U_quantizer_boundary[2**Q_bit_1], U_quantizer_level[2**Q_bit_1])
        Q_alphaR_2, Q_alphaI_2, Q_theta_2 = quantizer(alphaR_2, alphaI_2, theta_2, 2**Q_bit_2, 
                                                L_quantizer_boundary[2**Q_bit_2], L_quantizer_level[2**Q_bit_2], U_quantizer_boundary[2**Q_bit_2], U_quantizer_level[2**Q_bit_2])
        Q_alphaR = torch.cat((Q_alphaR_1, Q_alphaR_2), dim=1)
        Q_alphaI = torch.cat((Q_alphaI_1, Q_alphaI_2), dim=1)
        Q_theta = torch.cat((Q_theta_1, Q_theta_2), dim=1)
        
    Q_h = torch.zeros_like(h_act, dtype=torch.complex64).to(device) # (batch, M, K)
    from0toM = torch.arange(0, M, 1, dtype=torch.float32).to(device)
    Q_alpha = torch.complex(Q_alphaR, Q_alphaI).to(device)
    Q_theta = (torch.pi/180) * Q_theta
    
    for kk in range(2):
        for ii in range(paths):
            Q_theta_expended_temp = Q_theta[:, ii, kk].unsqueeze(1).repeat(1,M)
            Q_response_temp = torch.exp(1j * torch.pi * (torch.sin(Q_theta_expended_temp) * from0toM))
            Q_alpha_temp = Q_alpha[:, ii, kk].unsqueeze(1)
            Q_h[:, :, kk] += (Q_response_temp * Q_alpha_temp)
            

    ## Zero-forcing beamforming
    V_ZF = torch.linalg.pinv(H_hat) # (batch, M, K)
    V_ZF = V_ZF / torch.sqrt(torch.real(torch.einsum('bii->b', torch.bmm(V_ZF.mH , V_ZF)))).unsqueeze(1).unsqueeze(2)
    sumrate_ZF.append(rate_calc(K, V_ZF.to(device), h_act, noise_std_dl).item())

    ## Maximum Ratio Transmit beamforming
    V_MRT = H_hat.mH # (batch, M, K)
    V_MRT = V_MRT / torch.sqrt(torch.real(torch.einsum('bii->b', torch.bmm(V_MRT.mH , V_MRT)))).unsqueeze(1).unsqueeze(2)
    sumrate_MRT.append(rate_calc(K, V_MRT.to(device), h_act, noise_std_dl).item())
    
    ## Zero-forcing beamforming
    V_ZF = torch.linalg.pinv(Q_h.mT).to(device) # (batch, M, K)
    V_ZF = V_ZF / torch.sqrt(torch.real(torch.einsum('bii->b', torch.bmm(V_ZF.mH , V_ZF)))).unsqueeze(1).unsqueeze(2)
    ZF_lim.append(rate_calc(K, V_ZF.to(device), h_act, noise_std_dl).item())

    ## Maximum Ratio Transmit beamforming
    V_MRT = torch.conj(Q_h) # (batch, M, K)
    V_MRT = V_MRT / torch.sqrt(torch.real(torch.einsum('bii->b', torch.bmm(V_MRT.mH , V_MRT)))).unsqueeze(1).unsqueeze(2)
    MRT_lim.append( rate_calc(K, V_MRT.to(device), h_act, noise_std_dl).item())


torch.Size([10000, 4, 2])


In [84]:
print(sumrate_ZF)
print(sumrate_MRT)
print(ZF_lim)
print(MRT_lim)

[7.405674934387207, 6.415339469909668, 6.04055118560791, 5.804316997528076, 5.366418361663818, 5.330438137054443, 4.906224727630615]
[7.736385822296143, 6.6528239250183105, 6.185569763183594, 5.926987171173096, 5.468550682067871, 5.393192768096924, 4.9660115242004395]
[6.529465675354004, 3.868251085281372, 3.388394355773926, 2.2131507396698, 2.5207679271698, 2.5654711723327637, 2.5174548625946045]
[7.114041805267334, 4.214554309844971, 3.4130849838256836, 2.3959882259368896, 2.6083359718322754, 2.553652763366699, 2.549546718597412]


In [85]:
import pickle
with open('ZF OMP-CE inf', 'wb') as ff:
    pickle.dump(sumrate_ZF, ff)
with open('MRT OMP-CE inf', 'wb') as ff:
    pickle.dump(sumrate_MRT, ff)
with open('ZF OMP-CE lim', 'wb') as ff:
    pickle.dump(ZF_lim, ff)
with open('MRT OMP-CE lim', 'wb') as ff:
    pickle.dump(MRT_lim, ff)